# Practical Assignment 2: Q-Learning from Scratch 🧠
### Master's Degree Programme in Applications of AI to Industry and Communications
#### University of Alcalá

Welcome to your second practical assignment in the Reinforcement Learning module! In this notebook, **you will build your first Reinforcement Learning agent entirely from scratch**. 

Instead of relying on a high-level library like Stable-Baselines3, you will implement the foundational **Q-Learning** algorithm using pure Python and NumPy to solve the FrozenLake-v1 ❄️ and Taxi-v3 🚕 environments.

> [!TIP]
> **💡 Originality & Research Bonus:**
> Original solutions and deeper research are highly valued and rewarded in this practical! Students who investigate alternative exploration strategies (e.g., Softmax/Boltzmann exploration, Upper Confidence Bound exploration), experiment with dynamic learning rate decay schedules, test custom reward structures, or perform systematic hyperparameter comparisons will receive bonus marks. Show your initiative by exploring beyond the baseline!

Throughout this notebook, you will encounter interleaved theoretical questions. Please answer them in the designated markdown cells.

Let's get started!


## 1. Setting up the Computational Environment 🖥️
We will begin by installing the necessary dependencies.
- `gymnasium`: Provides the environments.
- `pygame`: Used for environment UI rendering.
- `numpy`: Used for matrix operations and our Q-table memory.


In [ ]:
!pip install -q gymnasium pygame numpy huggingface_hub imageio imageio_ffmpeg pyglet tqdm

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3-opengl ffmpeg xvfb
!pip install -q pyvirtualdisplay

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 2. Importing Packages 📦
In addition to the environment and math libraries, we import `random` for our stochastic policies and `imageio` to record replay videos.


In [ ]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import pickle
from tqdm.notebook import tqdm

# Set global random seed for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 3. Theoretical Foundations: Q-Learning 📚

Before writing code, let's review the mathematics of Q-Learning.

Q-Learning is a **Value-Based**, **Off-Policy** algorithm. Instead of learning the policy directly, it learns a *Q-Function* $Q(s, a)$. The Q-Function estimates the expected cumulative reward of taking action $a$ in state $s$ and acting optimally thereafter.

Since our state space and action space are both discrete and small, we can represent this Q-Function as a literal table (the **Q-Table**) with rows corresponding to states and columns corresponding to actions.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

### The Bellman Update Equation
As the agent explores the environment, it observes transitions: $(S_t, A_t, R_{t+1}, S_{t+1})$. It then updates its Q-Table using the Bellman Equation:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$

Where:
- $\alpha$ (Learning Rate): How much we accept the new information versus the old.
- $\gamma$ (Discount Factor): How much we care about future rewards versus immediate rewards.

---
### 📝 Question 1: The Bellman Equation Components
In the update equation above, there is a term inside the brackets: $\left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \right]$.
**This term is known as the TD Error (Temporal Difference Error). Explain in your own words what the TD Error represents conceptually. Why do we subtract the current $Q(S_t, A_t)$ from the target?**

*Your answer here...*


## 4. Part 1: Frozen Lake ⛄

Let's instantiate our first environment: [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/). The agent must navigate from Start (S) to Goal (G) by walking on frozen tiles (F) and avoiding holes (H).

We will start with the deterministic version: `is_slippery=False`.


In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")- `is_slippery=False`: The agent always moves in the intended direction due to the non-slippery nature of the frozen lake (deterministic).
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample()) # Get a random observation

The Observation Space is `Discrete(16)`. The state is represented as a single integer (0 to 15) indicating the grid position.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake">


In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action

The Action Space is `Discrete(4)`: LEFT (0), DOWN (1), RIGHT (2), UP (3).

The Reward Function is highly sparse:
- Reach goal: +1
- Reach hole: 0
- Reach frozen: 0

### Initializing the Q-Table
We need a table of size `(state_space, action_space)`. Let's initialize it with zeros.


In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

action_space = env.action_space.n
print("There are ", action_space, " possible actions")

In [ ]:
# Let's create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros. np.zeros needs a tuple (a,b)
def initialize_q_table(state_space, action_space):
  Qtable =
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

---
### 📝 Question 2: Q-Table Initialization
**Why is it common practice to initialize the Q-Table with zeros? Imagine we initialized the entire table with a large positive number (e.g., +100) instead. How would this alter the agent's exploratory behavior during the first few episodes?**

*Your answer here...*


## 5. Defining the Policies 🤖

Q-Learning is *off-policy*. We use an **$\epsilon$-greedy policy** to explore during training, but we learn the values of the strictly **greedy policy** (the $\max_a$ in the Bellman equation).

### The Greedy Policy (Exploitation)
This policy always selects the action with the highest Q-value for the given state.


In [ ]:
def greedy_policy(Qtable, state):
  # Exploitation: take the action with the highest state, action value
  action =

  return action

### The Epsilon-Greedy Policy (Exploration vs. Exploitation)
With probability $1 - \epsilon$, we exploit (use the greedy policy). With probability $\epsilon$, we explore (choose a completely random action).

Over time, we will decay $\epsilon$ so the agent explores less and exploits more.


In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num =
  # if random_num > greater than epsilon --> exploitation
  if random_num > epsilon:
    # Take the action with the highest value given a state
    # np.argmax can be useful here
    action =
  # else --> exploration
  else:
    action = # Take a random action

  return action

---
### 📝 Question 3: Epsilon Decay
**Explain the necessity of decaying $\epsilon$ over time rather than keeping it at a constant value (e.g., $\epsilon = 0.5$). What would happen to the agent's performance in the final stages of training if $\epsilon$ never decreased?**

*Your answer here...*


## 6. Training the Q-Learning Agent 🏃

Here are the hyperparameters for our FrozenLake training run.


In [ ]:
# Training parameters
n_training_episodes = 10000  # Total training episodes
learning_rate = 0.7          # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# Environment parameters
env_id = "FrozenLake-v1"     # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate
eval_seed = []               # The evaluation seed of the environment

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.05            # Minimum exploration probability
decay_rate = 0.0005            # Exponential decay rate for exploration prob

Now, implement the training loop. You will need to utilize your `epsilon_greedy_policy` to select actions, and implement the Bellman update equation manually inside the loop.


In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Choose the action At using epsilon greedy policy
      action =

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info =

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] =

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable

In [ ]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

Let's look at the resulting Q-table. You should see non-zero values indicating the expected returns of the paths discovered by the agent.


In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_frozenlake
}

## 7. Evaluating the Agent 📈
We will now evaluate the agent deterministically (using the greedy policy only).


In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for taxi-v3)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for taxi-v3)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

> [!IMPORTANT]
> **Stochastic Environments:**
> The non-slippery FrozenLake environment is deterministic, making it easy to achieve a mean reward of 1.0. In the real world, actions have uncertain outcomes. When `is_slippery=True`, the agent might slip and move perpendicular to its intended direction.

---
### 📝 Question 4: Handling Stochasticity
**In a stochastic environment (like the slippery FrozenLake), the agent does not always end up in the state it intended. How does the Q-learning algorithm implicitly handle this stochasticity without explicitly knowing or modeling the transition probabilities of the environment?**

*Your answer here...*


## 8. Part 2: Taxi-v3 - Independent Challenge 🚖

Now it is time to apply what you've learned to a much more complex environment. In [Taxi-v3](https://gymnasium.farama.org/environments/toy_text/taxi/), the agent must navigate to a passenger, pick them up, and drop them off at a designated location.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi">


In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

The state space is large (500 discrete states), encoding the taxi's position, the passenger's position, and the destination.


In [ ]:
# Create our Q table with state_size rows and action_size columns (500x6)
Qtable_taxi = initialize_q_table(state_space, action_space)
print(Qtable_taxi)
print("Q-table shape: ", Qtable_taxi .shape)

### Hyperparameter Tuning Challenge
For this environment, **you must determine the optimal hyperparameters yourself**. Think carefully about the number of states, how much exploration is needed, and the discount factor.

> [!TIP]
> - Since the state space is larger, you will likely need **more training episodes** than FrozenLake.
> - Since the episode can be long, $\gamma$ should be chosen carefully to propagate rewards backwards.
> - Make sure the decay rate allows the agent to explore long enough before $\epsilon$ becomes negligible.


In [ ]:
# Training parameters
# TODO: Define the number of training episodes and the learning rate
n_training_episodes =  # <YOUR_VALUE>
learning_rate =  # <YOUR_VALUE>

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# DO NOT MODIFY EVAL_SEED
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148]

# Environment parameters
env_id = "Taxi-v3"           # Name of the environment
max_steps = 99               # Max steps per episode
# TODO: Define the discount rate (gamma)
gamma =  # <YOUR_VALUE>

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
# TODO: Define minimum epsilon and decay rate
min_epsilon =  # <YOUR_VALUE>
decay_rate =  # <YOUR_VALUE>


In [ ]:
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)
Qtable_taxi

## 9. Assignment Delivery 🚀
You must publish your trained Taxi-v3 model to the Hugging Face Hub. To pass the assignment, you must achieve a `mean_reward >= 4.5` on the leaderboard evaluation.


In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json

In [ ]:
def record_video(env, Qtable, out_directory, fps=1):
  """
  Generate a replay video of the agent
  :param env
  :param Qtable: Qtable of our agent
  :param out_directory
  :param fps: how many frame per seconds (with taxi, we want to slow down)
  """
  images = []  
  terminated = False
  truncated = False
  state, info = env.reset(seed=random.randint(0,500))
  img = env.render()
  images.append(img)
  while not terminated and not truncated:
    # Take the action (index) that have the maximum expected future reward given that state
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action) # We directly put next_state = state for recording logic
    img = env.render()
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
def push_to_hub(
    repo_id, model, env, video_fps=1, local_repo_path="hub"
):
    """
    Evaluate, Generate a video and Upload a model to Hugging Face Hub.
    This method does the complete pipeline:
    - It evaluates the model
    - It generates the model card
    - It generates a replay video of the agent
    - It pushes everything to the Hub

    :param repo_id: repo_id: id of the model repository from the Hugging Face Hub
    :param env
    :param video_fps: how many frame per seconds to record our video replay
    (with taxi-v3 and frozenlake-v1 we use 1)
    :param local_repo_path: where the local repository is
    """
    _, repo_name = repo_id.split("/")

    eval_env = env
    api = HfApi()

    # Step 1: Create the repo
    repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
    )

    # Step 2: Download files
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Step 3: Save the model
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False

    # Pickle the model
    with open((repo_local_path) / "q-learning.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 4: Evaluate the model and build JSON with evaluation metrics
    mean_reward, std_reward = evaluate_agent(
        eval_env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"]
    )

    evaluate_data = {
        "env_id": model["env_id"],
        "mean_reward": mean_reward,
        "n_eval_episodes": model["n_eval_episodes"],
        "eval_datetime": datetime.datetime.now().isoformat()
    }

    # Write a JSON file called "results.json" that will contain the
    # evaluation results
    with open(repo_local_path / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")

    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-" + "no_slippery"

    metadata = {}
    metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

    # Add metrics
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
    )

    # Merges both dictionaries
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Q-Learning** Agent playing1 **{env_id}**
  This is a trained model of a **Q-Learning** agent playing **{env_id}** .

  ## Usage

  ```python

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # Don't forget to check if you need to add additional attributes (is_slippery=False etc)
  env = gym.make(model["env_id"])
  ```
  """

    evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

    readme_path = repo_local_path / "README.md"
    readme = ""
    print(readme_path.exists())
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)

    # Step 6: Record a video
    video_path = repo_local_path / "replay.mp4"
    record_video(env, model["qtable"], video_path, video_fps)

    # Step 7. Push everything to the Hub
    api.upload_folder(
        repo_id=repo_id,
        folder_path=repo_local_path,
        path_in_repo=".",
    )

    print("Your model is pushed to the Hub. You can view your model here: ", repo_url)

1️⃣ Authenticate with Hugging Face.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

2️⃣ Push the Taxi-v3 model to the Hub.


In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_taxi
}

In [ ]:
username = "" # FILL THIS
repo_name = "" # FILL THIS
push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env)

## 10. Visualizing the Learning Progression: Beginning & Final 🎥

Let's observe the evolution of the Q-Table in the Taxi-v3 environment:
1. **Beginning (Untrained):** Initial empty Q-table (all zeros); the taxi takes random decisions.
2. **Final (Fully Trained):** Converged Q-table; executes the optimal path directly to pickup and drop-off.

In [ ]:
import imageio
import base64
import numpy as np
from IPython.display import HTML

def record_taxi_video(Q_matrix, filename="taxi_replay.mp4", max_ep_steps=100, seed=42):
    env_eval = gym.make("Taxi-v3", render_mode="rgb_array")
    state, info = env_eval.reset(seed=seed)
    frames = []
    total_r = 0
    for _ in range(max_ep_steps):
        frames.append(env_eval.render())
        action = np.argmax(Q_matrix[state][:])
        state, reward, terminated, truncated, info = env_eval.step(action)
        total_r += reward
        if terminated or truncated:
            break
    env_eval.close()
    imageio.mimsave(filename, frames, fps=5)
    print(f"🎬 Saved: {filename} (Total Reward: {total_r:.1f})")
    return filename

# 1. Record Untrained Agent (all zeros Q-table)
Qtable_empty = np.zeros_like(Qtable_taxi)
record_taxi_video(Qtable_empty, "taxi_untrained.mp4", seed=SEED)

# 2. Record Fully Trained Agent
record_taxi_video(Qtable_taxi, "taxi_final.mp4", seed=SEED)

def get_b64(path):
    return base64.b64encode(open(path, "rb").read()).decode("utf-8")

b64_taxi_untrained = get_b64("taxi_untrained.mp4")
b64_taxi_final = get_b64("taxi_final.mp4")

HTML(f"""
<div style="font-family: Arial, sans-serif; text-align: center; margin: 15px 0;">
  <h3 style="color: #1a1a1a;">🚕 Taxi-v3 Policy Evolution</h3>
  <div style="display: flex; justify-content: space-around; flex-wrap: wrap; gap: 15px; margin-top: 15px;">
    
    <!-- Untrained -->
    <div style="flex: 1; min-width: 260px; background: #fff1f0; border: 1px solid #ffa39e; border-radius: 8px; padding: 12px;">
      <h4 style="color: #cf1322; margin-top: 0;">1. Beginning (Untrained) 💥</h4>
      <video width="100%" height="240" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_taxi_untrained}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Untrained:</b> Random navigation, wall collisions, timeout with negative reward.</p>
    </div>

    <!-- Final -->
    <div style="flex: 1; min-width: 260px; background: #f6ffed; border: 1px solid #b7eb8f; border-radius: 8px; padding: 12px;">
      <h4 style="color: #389e0d; margin-top: 0;">2. Final (Fully Trained) 🚕🏆</h4>
      <video width="100%" height="240" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_taxi_final}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Fully Trained:</b> Direct shortest path to passenger pickup and destination drop-off (+8.5 reward).</p>
    </div>

  </div>
</div>
""")

## 11. Automated Tournament & Competition Submission 🏆

To participate in the **class tournament** and submit your practical for grading:

1. **Package your agent:** Run the cells below to save `q-learning.pkl` and generate `agent.py` into `submission.zip`.
2. **Submit to the Automated GitHub Tournament:**
   - Create a folder with your GitHub username: `submissions/practical2/<your_github_username>/`
   - Copy `agent.py` and `q-learning.pkl` into that folder.
   - Open a **Pull Request (PR)** against the repository.
   - An automated **GitHub Actions bot** will evaluate your agent across 50 benchmark episodes and post your official score card directly as a comment on your PR within 2 minutes!
   - Once merged, your model will be ranked on the live [LEADERBOARD.md](https://github.com/gramuah/RL_tournament/blob/main/LEADERBOARD.md) 🥇🥈🥉.
3. **Submit to Virtual Campus (Blackboard):** Download the generated `submission.zip` and your completed `.ipynb` notebook and upload them to Blackboard before the deadline.

In [ ]:
import pickle

# Save our model dictionary to a file
with open("q-learning.pkl", "wb") as f:
    pickle.dump(model, f)


In [ ]:
%%writefile agent.py
import os
import numpy as np
import pickle

# Load the model
model_path = os.path.join(os.path.dirname(__file__), "q-learning.pkl")
with open(model_path, "rb") as f:
    model = pickle.load(f)
    
Qtable = model["qtable"]

def agent(observation):
    # Select the action with the maximum expected reward
    action = np.argmax(Qtable[observation][:])
    return int(action)


In [ ]:
!zip submission.zip q-learning.pkl agent.py

### 🚀 Option A: Submit Pull Request Directly from this Notebook (Recommended)

You can automatically fork the repository, upload your model, and open a **Pull Request** to the tournament leaderboard with a single click!

> **Prerequisite:** You need a **GitHub Personal Access Token (PAT)** with `repo` permissions.
> 1. Go to [github.com/settings/tokens](https://github.com/settings/tokens).
> 2. Click **Generate new token (classic)**, check the **`repo`** checkbox, and click **Generate token**.
> 3. Paste it in the prompt below (it will be masked securely).

In [ ]:
import getpass
import urllib.request

# 1. Load the automated submission tool from the repository
exec(urllib.request.urlopen("https://raw.githubusercontent.com/gramuah/RL_tournament/main/tournament/submit.py").read().decode("utf-8"))

# 2. Enter your GitHub Token securely
GITHUB_TOKEN = getpass.getpass("🔑 Enter your GitHub Personal Access Token: ")

# 3. Automatically fork, upload, and open the Pull Request!
pr_url = submit_practical(
    practical_id=2,
    files_to_submit={
        "agent.py": "agent.py",
        "q-learning.pkl": "q-learning.pkl"
    },
    github_token=GITHUB_TOKEN
)

### 🥊 Head-to-Head Agent Duel (Compara tu agente contra otro alumno o baseline)

¿Quieres ver **quién aterriza más rápido en Taxi-v4** o cómo se enfrenta tu agente al mismo desafío exacto que el de un compañero?

Puedes comparar tu modelo y ver las visualizaciones de tres formas distintas:

1. **🌐 En la Arena Web Interactiva de GitHub Pages (1 Clic, sin código):**
   Entra en el portal público oficial: **[https://gramuah.github.io/RL_tournament/](https://gramuah.github.io/RL_tournament/)**.
   Permite seleccionar a cualquier compañero de la clase o baseline en desplegables y ver la **retransmisión en vídeo sincronizado split-screen**, telemetría en tiempo real y el podio de velocidad.

2. **⚙️ Lanzar Duelos a la Carta en GitHub Actions:**
   Desde la pestaña **[Actions > Run Agent Duel](https://github.com/gramuah/RL_tournament/actions/workflows/duel.yml)** puedes pulsar *"Run workflow"* para que el bot de GitHub simule el enfrentamiento y publique el GIF animado en el resumen de la ejecución.

3. **🎬 Animaciones automáticas en [LEADERBOARD.md](https://github.com/gramuah/RL_tournament/blob/main/LEADERBOARD.md):**
   El ranking general de la clase incluye animaciones GIF incrustadas que se reproducen directamente en la propia página de GitHub.

4. **💻 Directamente en este Notebook (código a continuación):**
   Ejecuta la celda inferior para cargar el comparador, enfrentar tu `agent.py` contra cualquier oponente (baseline o usuario de GitHub de un compañero) y visualizar el vídeo sincronizado aquí mismo en Colab.

In [ ]:
# ==============================================================================
# 🥊 HEAD-TO-HEAD AGENT DUEL (Taxi-v3 / Taxi-v4)
# ==============================================================================
import urllib.request

# 1. Descarga e importa el comparador oficial
exec(urllib.request.urlopen("https://raw.githubusercontent.com/gramuah/RL_tournament/main/tournament/compare.py").read().decode("utf-8"))

# 2. Elige tu contrincante:
# - Baseline disponible: "baseline_random"
# - O escribe el usuario de GitHub de cualquier compañero que haya subido su modelo!
OPPONENT = "baseline_random"

# 3. Ejecuta el enfrentamiento (ambos con la misma semilla y configuración inicial)
duel_stats = duel_agents(
    practical_id=2,
    agent_a="agent.py",           # Tu agente entrenado
    agent_b=OPPONENT,             # Tu oponente
    episodes=5,                   # Número de carreras cara a cara
    seed=42,
    output_video="duel_p2.mp4"
)

# 4. Reproduce el vídeo sincronizado directamente aquí en Colab
display_duel("duel_p2.mp4")


### 📁 Option B: Deliver Deliverables to Virtual Campus (Blackboard)

Remember that to complete your course deliverable, you must also:
1. Download your finished notebook (`.ipynb`) with all cell outputs visible (`File > Download > Download .ipynb`).
2. Download the `submission.zip` generated above.
3. Submit both files through the **Blackboard** assignment box before the deadline.

________________________________________________________________________
**Congratulations on completing your Q-Learning Master's assignment!** 

You have implemented a foundational RL algorithm entirely from scratch, handled the exploration-exploitation trade-off, tuned hyperparameters independently for a complex environment, and tackled theoretical questions regarding Bellman equations and stochasticity. 

Ensure your inline questions are answered and your model is uploaded to the Hub before finalizing your submission.
